# Exploratory Data Analysis: Container Freight Delay Dataset

Run `python src/generate_data.py` and `python src/features.py` before opening this notebook.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

raw = pd.read_csv('../data/raw_shipments.csv', parse_dates=['departure_date', 'scheduled_arrival', 'actual_arrival'])
feat = pd.read_csv('../data/features.csv', parse_dates=['departure_date'])
print(f'Raw: {raw.shape}  Features: {feat.shape}')

## 1. Dataset overview

In [ ]:
print('=== Raw shipments ===')
print(raw.dtypes)
print('\nDelay rate:', raw['delayed_24h'].mean().round(4))
print('\nDate range:', raw['departure_date'].min().date(), 'to', raw['departure_date'].max().date())

In [ ]:
raw.describe().round(2)

## 2. Target variable distribution

In [ ]:
by_year = raw.groupby(raw['departure_date'].dt.year)['delayed_24h'].mean().reset_index()
by_year.columns = ['year', 'delay_rate']
px.bar(by_year, x='year', y='delay_rate', title='Delay rate by year',
       labels={'delay_rate': 'Delay rate'}, template='plotly_white')

In [ ]:
monthly = raw.assign(month=raw['departure_date'].dt.to_period('M')).groupby('month')['delayed_24h'].mean()
monthly.index = monthly.index.to_timestamp()
fig = go.Figure(go.Scatter(x=monthly.index, y=monthly.values * 100, mode='lines+markers',
                           line=dict(color='#2563EB', width=2)))
fig.add_vrect(x0='2020-01-01', x1='2022-12-31', fillcolor='rgba(220,38,38,0.08)', line_width=0,
              annotation_text='COVID era', annotation_font_color='#DC2626')
fig.update_layout(title='Monthly delay rate (2019-2024)', xaxis_title='Month',
                  yaxis_title='Delay rate (%)', template='plotly_white')
fig.show()

## 3. Delay drivers: univariate analysis

In [ ]:
px.box(raw, x='delayed_24h', y='dest_congestion_index',
       color='delayed_24h', title='Destination congestion vs delay outcome',
       template='plotly_white', labels={'delayed_24h': 'Delayed >24h'})

In [ ]:
px.box(raw, x='delayed_24h', y='vessel_utilization_pct',
       color='delayed_24h', title='Vessel utilization vs delay outcome',
       template='plotly_white')

In [ ]:
carrier_stats = raw.groupby('carrier_name').agg(
    delay_rate=('delayed_24h', 'mean'),
    n=('delayed_24h', 'count')
).reset_index().sort_values('delay_rate', ascending=False)
px.bar(carrier_stats, x='carrier_name', y='delay_rate', title='Delay rate by carrier',
       template='plotly_white', labels={'delay_rate': 'Delay rate'})

In [ ]:
px.box(raw, x='delayed_24h', y='weather_severity_score',
       color='delayed_24h', title='Weather severity vs delay outcome',
       template='plotly_white')

## 4. Port-level analysis

In [ ]:
dest_delay = raw.groupby('destination_port')['delayed_24h'].mean().sort_values(ascending=False).reset_index()
px.bar(dest_delay, x='destination_port', y='delayed_24h', title='Delay rate by destination port',
       template='plotly_white', labels={'delayed_24h': 'Delay rate'})

In [ ]:
route_stats = raw.groupby('route_id').agg(
    delay_rate=('delayed_24h', 'mean'), n=('delayed_24h', 'count')
).query('n >= 100').nlargest(20, 'delay_rate').reset_index()
px.bar(route_stats.sort_values('delay_rate'), x='delay_rate', y='route_id',
       orientation='h', title='Top 20 routes by delay rate',
       template='plotly_white', labels={'delay_rate': 'Delay rate'})

## 5. Engineered feature distributions

In [ ]:
px.histogram(feat, x='route_delay_rate_30d', color='delayed_24h',
             nbins=50, barmode='overlay', opacity=0.7,
             title='30-day route delay rate distribution by outcome',
             template='plotly_white')

In [ ]:
px.histogram(feat, x='carrier_ontime_rate_30d', color='delayed_24h',
             nbins=50, barmode='overlay', opacity=0.7,
             title='Carrier 30-day on-time rate by outcome',
             template='plotly_white')

## 6. Correlation with delay (Spearman)

In [ ]:
from features import FEATURE_COLS, TARGET_COL
corr = feat[FEATURE_COLS + [TARGET_COL]].corr(method='spearman')[TARGET_COL].drop(TARGET_COL).sort_values()
fig = go.Figure(go.Bar(
    x=corr.values, y=corr.index, orientation='h',
    marker_color=['#DC2626' if v > 0 else '#16A34A' for v in corr.values]
))
fig.update_layout(title='Spearman correlation with delayed_24h',
                  xaxis_title='Correlation', template='plotly_white',
                  height=700, margin=dict(l=200))
fig.show()

## 7. Temporal split preview

In [ ]:
for label, year_range in [('Train', range(2019, 2023)), ('Val', [2023]), ('Test', [2024])]:
    subset = feat[feat['year'].isin(year_range)]
    print(f'{label:10}  n={len(subset):>7,}  delay_rate={subset["delayed_24h"].mean():.2%}')